In [1]:
pip install opacus

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from torchvision import datasets
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import optim
from opacus import PrivacyEngine
from torch.autograd import Variable
import numpy as np

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

In [5]:
train_data = CIFAR10(
    root = 'data',
    train = True,
    transform = transform,
    download = True,
)
test_data = CIFAR10(
    root = 'data',
    train = False,
    transform = transform
)

In [6]:
loaders = {
    'train' : torch.utils.data.DataLoader(train_data,
                                          batch_size=100,
                                          shuffle=True,),

    'test'  : torch.utils.data.DataLoader(test_data,
                                          batch_size=100,
                                          shuffle=True,),
}

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out = nn.Linear(64 * 4 * 4, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output, x

'class CNN(nn.Module):\n    def __init__(self):\n        super(CNN, self).__init__()\n        self.conv1 = nn.Sequential(\n            nn.Conv2d(3, 16, 5, 1, 2),\n            nn.ReLU(),\n            nn.MaxPool2d(2),\n        )\n        self.conv2 = nn.Sequential(\n            nn.Conv2d(16, 32, 5, 1, 2),\n            nn.ReLU(),\n            nn.MaxPool2d(2),\n        )\n        self.conv3 = nn.Sequential(\n            nn.Conv2d(32, 64, 5, 1, 2),\n            nn.ReLU(),\n            nn.MaxPool2d(2),\n        )\n        self.out = nn.Linear(64 * 4 * 4, 10)\n    def forward(self, x):\n        x = self.conv1(x)\n        x = self.conv2(x)\n        x = self.conv3(x)\n        x = x.view(x.size(0), -1)\n        output = self.out(x)\n        return output, x'

In [9]:
cnn = CNN()

In [10]:
cnn

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 96, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 96, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(96, 80, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (7): ReLU()
    (8): Conv2d(80, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (9): ReLU()
    (10): Conv2d(64, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (11): ReLU()
    (12): Conv2d(64, 96, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (13): ReLU()
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6144, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)

In [11]:
lossf = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr = 0.001)
def acc(predictions,labels):
    return (predictions==labels).mean()

In [12]:
privacy_engine = PrivacyEngine()
cnn, optimizer, loaders['train'] = privacy_engine.make_private_with_epsilon(
    module=cnn,
    optimizer=optimizer,
    data_loader=loaders['train'],
    epochs=25,
    target_epsilon=8,
    target_delta=1e-5,
    max_grad_norm=1.0,
)

/opt/anaconda3/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


In [13]:
num_epochs = 25
def train(num_epochs, cnn, loaders):
    cnn.train()
    total_step = len(loaders['train'])
    for epoch in range(num_epochs):
        overall_acc = []
        overall_loss = []
        for i, (images, labels) in enumerate(loaders['train']):
            b_x = Variable(images)
            b_y = Variable(labels)
            output = cnn(b_x)[0]
            predictions = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = labels.detach().cpu().numpy()
            accuracy = acc(predictions,labels)
            overall_acc.append(accuracy)
            loss = lossf(output, b_y)
            overall_loss.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if (i+1) % 500 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Accuracy: {:.4f}, Loss: {:.4f}'.format(epoch + 1, num_epochs, i + 1, total_step, np.mean(overall_acc) * 100, np.mean(overall_loss)))
                pass
            pass
        pass

In [14]:
train(num_epochs, cnn, loaders)

Epoch [1/25], Step [500/500], Accuracy: 23.5409, Loss: 2.0655
Epoch [2/25], Step [500/500], Accuracy: 36.6418, Loss: 1.8440
Epoch [3/25], Step [500/500], Accuracy: 40.4134, Loss: 2.0612
Epoch [4/25], Step [500/500], Accuracy: 41.3024, Loss: 2.3315
Epoch [5/25], Step [500/500], Accuracy: 42.1434, Loss: 2.4852
Epoch [6/25], Step [500/500], Accuracy: 41.6959, Loss: 2.6091
Epoch [7/25], Step [500/500], Accuracy: 42.1887, Loss: 2.6768
Epoch [8/25], Step [500/500], Accuracy: 41.9349, Loss: 2.6394
Epoch [9/25], Step [500/500], Accuracy: 42.2799, Loss: 2.5812
Epoch [10/25], Step [500/500], Accuracy: 42.7694, Loss: 2.5998
Epoch [11/25], Step [500/500], Accuracy: 43.0801, Loss: 2.6586
Epoch [12/25], Step [500/500], Accuracy: 43.2721, Loss: 2.7279
Epoch [13/25], Step [500/500], Accuracy: 42.8559, Loss: 2.7430
Epoch [14/25], Step [500/500], Accuracy: 42.8683, Loss: 2.7252
Epoch [15/25], Step [500/500], Accuracy: 42.9261, Loss: 2.6418
Epoch [16/25], Step [500/500], Accuracy: 43.4130, Loss: 2.5284
E

In [15]:
def test():
    cnn.eval()
    overall_acc = []
    overall_loss = []
    with torch.no_grad():
        for images, labels in loaders['test']:
            test_output, last_layer = cnn(images)
            pred_y = torch.max(test_output, 1)[1].data.squeeze()
            labels = Variable(labels)
            loss = lossf(test_output, labels)
            overall_loss.append(loss.item())
            accuracy = (pred_y == labels).sum().item() / float(labels.size(0))
            labels = labels.detach().cpu().numpy()
            overall_acc.append(accuracy)
            pass
        print('Test Accuracy: %.4f' % (np.mean(overall_acc) * 100) + ', Test Loss: %.4f' % np.mean(overall_loss))
    pass

In [16]:
test()

Test Accuracy: 44.5100, Test Loss: 2.3722
